In [0]:
# Read the Control Table
control_df = spark.read.table(
    "control_catalog.metadata.pipeline_control"
)

# Keep only active records and collect them for looping
active_rows = control_df.filter("active = true").collect()

# Process each active record one by one
for row in active_rows:

    # Get the source file path
    source_path = row["source_path"]

    # Build the complete target table name
    target_table = (
        f"{row['target_catalog']}."
        f"{row['target_schema']}."
        f"{row['target_table']}"
    )

    # Display the source and target
    print(f"Reading: {source_path}")
    print(f"Writing to: {target_table}")

    # Read the source Parquet file
    df = spark.read.format("parquet").load(source_path)

    # Write the data as a Delta table, replacing existing data
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    # Confirm the target table was completed
    print(f"Completed: {target_table}")